# C2 · Wrong labels: boosting chases them, the forest does not

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Corrupt 20% of the training labels and measure the accuracy on a **clean** test set as a function of the XGBoost rounds (learning rate 0.1, `max_depth=5`): it rises up to about 25 rounds and then drops by **about 6 points** while a random forest stays stable. Then show that regularization (`min_child_weight`,
`reg_lambda`, `subsample`, shallow trees) limits the damage.

## Dataset

`make_classification(3000, 10 features, 5 informative, flip_y=0)` clean, 60/40 split (`X_tr, X_te, y_tr, y_te`). `y_noisy` = training labels with 20% swapped at random (`rng = np.random.default_rng(0)`). The test set stays clean.

## Educational purpose

Understand the difference in temperament between forest and boosting: the forest averages independent trees and the label errors cancel out; boosting **always corrects the errors of the previous ones**, so with enough rounds it tries to "explain" the wrong labels too.

## Task

1. Train `XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=5, eval_metric="logloss", n_jobs=-1, random_state=0)` on `X_tr, y_noisy` and compute with `iteration_range=(0, k)` the accuracy on the clean test set for `k` in `[10, 25, 50, 100, 250, 500]`: `xgb_acc` (`pd.Series`).
2. Repeat with `XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=3, min_child_weight=20, reg_lambda=10, subsample=0.7, ...)`: `reg_xgb_acc`. Also train `RandomForestClassifier(300, random_state=0, n_jobs=-1)` on `y_noisy` and save `rf_acc`, and the same accuracy with clean labels `clean_xgb_acc` (base XGB at 500 rounds on `y_tr`).
3. Save `xgb_peak` (`xgb_acc.max()`), `peak_k`, `xgb_loss = xgb_peak − xgb_acc[500]` and `reg_loss` (same for `reg_xgb_acc`).
4. Save `xgb_beaten_by_rf` (True if `rf_acc` is larger than `xgb_acc[500]`) and `rf_beats_xgb_at_peak` (True if `rf_acc` is larger than `xgb_peak`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(3000, n_features=10, n_informative=5, class_sep=1.2, flip_y=0, random_state=2)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
rng = np.random.default_rng(0)
y_noisy = np.where(rng.random(len(y_tr)) < 0.2, 1 - y_tr, y_tr)
ks = [10, 25, 50, 100, 250, 500]

## Your work

In [ ]:
xgb_acc = pd.Series(dtype=float)
reg_xgb_acc = pd.Series(dtype=float)
rf_acc = None
clean_xgb_acc = None

In [ ]:
xgb_peak = None
peak_k = None
xgb_loss = None
reg_loss = None
xgb_beaten_by_rf = None
rf_beats_xgb_at_peak = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`m = xgb.XGBClassifier(...).fit(X_tr, y_noisy)`; `xgb_acc[k] = (m.predict(X_te, iteration_range=(0, k)) == y_te).mean()`. Forest: `RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(X_tr, y_noisy).score(X_te, y_te)`.'):
    check(list(xgb_acc.index) == ks and list(reg_xgb_acc.index) == ks, "curves over six values of k", "iteration_range=(0, k)")
    check(clean_xgb_acc > 0.93, "with clean labels XGBoost reaches over 0.93", "y_tr instead of y_noisy")
    check(peak_k in (25, 50, 100) and xgb_peak > 0.89 and xgb_loss > 0.04 and xgb_acc[500] < xgb_acc[peak_k] - 0.04, "with noisy labels XGBoost has a peak at 25-100 rounds and then loses over 4 points", "xgb_acc.max() and xgb_acc[500]")
    check(reg_loss < xgb_loss - 0.02 and reg_xgb_acc.max() > 0.9, "the regularized model loses less than the unregularized one (by over 2 points) and goes over 0.9 at the peak", "min_child_weight, reg_lambda, subsample, depth 3")
    check(rf_acc > 0.9 and xgb_beaten_by_rf is True, "the forest (over 0.9) beats the unregularized XGBoost at 500 rounds", "rf_acc > xgb_acc[500]")
    check(rf_beats_xgb_at_peak in (True, False), "boolean answer (the forest and the XGBoost peak are close)", "compare rf_acc and xgb_peak")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_noisy_labels_boosting_chases_them.ipynb`. Open it after you pass the check, or after you have really tried.